# Loading Raw Data In and Importing Libraries

In [49]:
import pandas as pd
import networkx as nx
import os
import pathlib
import re
from scipy import stats
import itertools
import matplotlib.pyplot as plt

def list_files_in_directory(directory_path:str) -> list[str]:
    """returns file paths in raw_data/"""
    
    files_list = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path,entry)
        if os.path.isfile(full_path):
            files_list.append(full_path)
    return files_list

def create_data_dict(path_li:list[str]) -> dict[str,pd.DataFrame]:
    """ makes a dictionary containing dataframes """
    
    data_dict = {}
    for path in path_li:
        file_name = pathlib.PurePath(path).name.split('.')[0]
        df = pd.read_csv(path)
        data_dict[file_name] = df
    
    return data_dict

# loading all data in raw_data/
DATA_DIR = os.path.join(os.getcwd(),'raw_data')
PATH_LI = list_files_in_directory(DATA_DIR)
DATA_DICT = create_data_dict(PATH_LI)

# splitting data into separate dataframes
medals = DATA_DICT['olympic_medals']
results = DATA_DICT['olympic_results']
athletes = DATA_DICT['olympic_athletes']
hosts = DATA_DICT['olympic_hosts']
uscities = DATA_DICT['uscities']
athlete_profiles = DATA_DICT['athlete_profiles']

# Medal Winning Athlete Biographic Modeling
Using biographical details regarding team USA athletes, we construct a network graph of associated features. This analysis is to help us determine the importance of these features when it comes to winning medals.

In [52]:

## cleaning athlete_profiles ##
# Dropping unsuccessful scrapes
usa_athletes = DATA_DICT['athlete_profiles']
remove_unknown_str = "hometown != 'Unknown' and height != 'Unknown' and age != 'Unknown' and education != 'Unknown'"
usa_athletes = usa_athletes.query(remove_unknown_str)
percent_kept = (len(usa_athletes) / len(DATA_DICT['athlete_profiles'])) * 100
# print(f"Percentage of athletes kept after removing 'Unknown' values: {percent_kept:.2f}%")

# Extracting city and state from hometown
def extract_city_state(hometown:str) -> tuple[str, str]:
    """Extracts city and state from a hometown string."""
    match = re.match(r"^(.*),\s*([A-Z]{2})$", hometown)
    if match:
        city = match.group(1).strip()
        state = match.group(2).strip()
        return city, state
    else:
        # Look up the city in uscities to find the state
        city_match = uscities[uscities['city'].str.lower() == hometown.strip().lower()]
        if not city_match.empty:
            return hometown.strip(), city_match.iloc[0]['state_id']
        return None, None

usa_athletes['city'], usa_athletes['state'] = zip(*usa_athletes['hometown'].apply(extract_city_state))

# Changing height to decimal feet
def parse_height_to_feet(height_str:str) -> float:
    """Converts a height string like 5'6\" into decimal feet."""
    match = re.match(r"^\s*(\d+)'\s*(\d+)\"\s*$", str(height_str))
    if not match:
        return None
    feet = int(match.group(1))
    inches = int(match.group(2))
    return feet + (inches / 12)

usa_athletes['height'] = usa_athletes['height'].apply(parse_height_to_feet).astype(float)

# changing age to int and making deceased a binary column
def parse_age_and_deceased(age_str:str) -> tuple[int, int]:
    """Extracts integer age and deceased flag from age text."""
    age_text = str(age_str)
    deceased = 1 if re.search(r"died", age_text, flags=re.IGNORECASE) else 0
    age_match = re.search(r"(\d+)", age_text)
    age_value = int(age_match.group(1)) if age_match else None
    return age_value, deceased

age_parsed = usa_athletes['age'].apply(parse_age_and_deceased)
usa_athletes['age'] = [value[0] for value in age_parsed]
usa_athletes['deceased'] = [value[1] for value in age_parsed]
usa_athletes['age'] = usa_athletes['age'].astype('Int64')
usa_athletes

# merging with medal data and keeping relevant columns
athlete_bio_medals = pd.merge(usa_athletes,
                              medals,
                              on='athlete_full_name',
                              how='inner')
bio_medal_columns = ['athlete_full_name', 'age', 'deceased', 'height', 'city', 'state', 'medal_type','event_title']
athlete_bio_medals = athlete_bio_medals[bio_medal_columns]
display(athlete_bio_medals)


# generating groupings for edges of graph
state_g = nx.Graph()
for name, group in athlete_bio_medals.groupby(by=['state']):
    print(f"Edge: {name}")

    # print(f"Number of states: {len(group.state.unique())}")
    # states = group.state.unique()
    # state_edges = list(itertools.combinations(states,2))
    # print(state_edges)
    print(group[['state', 'medal_type', 'event_title']])
    print()






,athlete_full_name,age,deceased,height,city,state,medal_type,event_title
0,Alex FERREIRA,31,0,5.750000,Aspen,CO,BRONZE,Men's Freeski Halfpipe
1,Alex FERREIRA,31,0,5.750000,Aspen,CO,SILVER,Men’s Ski Halfpipe
2,Chloe KIM,25,0,5.250000,Torrance,CA,GOLD,Women's Snowboard Halfpipe
3,Chloe KIM,25,0,5.250000,Torrance,CA,GOLD,Ladies’ Halfpipe
4,Nick BAUMGARTNER,44,0,6.000000,Iron River,MI,GOLD,Mixed Team Snowboard Cross
...,...,...,...,...,...,...,...,...
255,John Hamann NUNN,48,0,6.166667,San Diego,CA,BRONZE,double sculls 2x men
256,Lynn Alfred III WILLIAMS,32,0,5.583333,Fresno,CA,SILVER,twoperson keelboat open Star mixed
257,Marcia Ingram JONES-SMOKE,84,0,5.500000,Oklahoma City,OK,BRONZE,K1 500m kayak single women
258,Walter Francis DAVIS,46,0,6.166667,Arnaudville,LA,GOLD,high jump men


Edge: ('AK',)
   state medal_type               event_title
61    AK       GOLD  Ladies’ Team Sprint Free

Edge: ('AL',)
    state medal_type            event_title
199    AL     BRONZE  100m backstroke women
200    AL     SILVER  200m backstroke women
246    AL       GOLD        two-woman women
248    AL       GOLD       pole vault women

Edge: ('AR',)
    state medal_type     event_title
18     AR     SILVER      Trap women
183    AR     BRONZE  pole vault men

Edge: ('AZ',)
    state medal_type                             event_title
74     AZ     SILVER  synchronized diving 3m springboard men
109    AZ       GOLD                     mixed doubles mixed
227    AZ     BRONZE                      400m freestyle men
228    AZ     BRONZE                      400m freestyle men
249    AZ       GOLD                          pole vault men

Edge: ('CA',)
    state medal_type                             event_title
2      CA       GOLD              Women's Snowboard Halfpipe
3      CA      